In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt

from regpy.vecsps import UniformGridFcts
from regpy.operators import GaussianBlur
from regpy.hilbert import L2
from regpy.solvers import Setting
from regpy.solvers.linear import FISTA as LinearFISTA
from regpy.solvers.linear.spectral import (
    Spectral_Landweber,
    Spectral_IteratedTikhonov,
)
from regpy.solvers.nonlinear import (
    Landweber as NonlinearLandweber,
    FISTA as GeneralizedFISTA,
    IrgnmCG,
    LevenbergMarquardt,
)
import regpy.solvers.linear as linear_solvers

In [ ]:
grid = UniformGridFcts((-1, 1, 128), periodic=True)
points = grid.coords[0]

exact = (
    1.5
    + 0.4 * np.cos(np.pi * points)
    + 0.2 * np.sin(4 * np.pi * points)
)
initial = 0.8 + 0.1 * np.cos(2 * np.pi * points)

def make_operator():
    return GaussianBlur(grid, sigma=0.08, self_adjoint=False)

data = make_operator()(exact).copy()

def make_setting(regpar=None):
    # Separate operators avoid sharing mutable evaluation buffers.
    return Setting(
        op=make_operator(),
        penalty=L2,
        data_fid=L2,
        data=data.copy(),
        init=initial.copy(),
        regpar=regpar,
        codomain_free=False,
    )

plt.plot(points, exact, label="Exact solution")
plt.plot(points, data, label="Blurred data")
plt.plot(points, initial, "--", label="Nonzero initial guess")
plt.legend()
plt.show()

In [ ]:
comparisons = []

def take_steps(solver, number):
    snapshots = []
    for step in range(number):
        if not solver.next():
            raise AssertionError(
                f"{type(solver).__name__} stopped before step {step + 1}"
            )
        snapshots.append((
            solver.x.copy(),
            None if solver.y is None else solver.y.copy(),
        ))
    return snapshots

def relative_error(actual, reference):
    denominator = max(
        np.linalg.norm(reference),
        np.finfo(float).tiny,
    )
    return np.linalg.norm(actual - reference) / denominator

def compare(label, left, right, tolerance):
    errors_x = [
        relative_error(a[0], b[0])
        for a, b in zip(left, right, strict=True)
    ]
    errors_y = [
        relative_error(a[1], b[1])
        for a, b in zip(left, right, strict=True)
    ]

    print(label)
    print("  relative x differences:", np.array(errors_x))
    print("  relative y differences:", np.array(errors_y))

    assert max(errors_x) <= tolerance, (label, errors_x)
    assert max(errors_y) <= tolerance, (label, errors_y)
    comparisons.append((label, left[-1][0], right[-1][0]))

# Tight inner solves make the Newton comparisons meaningful.
cg_pars = {
    "tol": 1e-10,
    "reltolx": 1e-10,
    "reltoly": None,
    "max_it": 1000,
}

In [ ]:
nonlinear = NonlinearLandweber(
    make_setting(),
    stepsize=1.0,
    logging_level="WARNING",
)
spectral = Spectral_Landweber(
    make_setting(),
    k=[1, 2, 3],
    stepsize=1.0,
)

compare(
    "Landweber: nonlinear versus spectral",
    take_steps(nonlinear, 3),
    take_steps(spectral, 3),
    tolerance=1e-8,
)

In [ ]:
alpha = 1e-2
fixed_tau = 0.9
number_of_steps = 30

generalized = GeneralizedFISTA(
    make_setting(regpar=alpha),
    stepsize_selection_steps=[0],  # No subsequent norm estimation
    logging_level="WARNING",
)

# With finite-Lipschitz fidelity, genFISTA ignores constructor tau.
# Set the fixed step size explicitly and reset its acceleration state.
generalized.tau = fixed_tau
generalized._restart_three_term_recursion()

linear = LinearFISTA(
    make_setting(regpar=alpha),
    tau=fixed_tau,
    logging_level="WARNING",
)

compare(
    "FISTA versus genFISTA — fixed step size",
    take_steps(linear, number_of_steps),
    take_steps(generalized, number_of_steps),
    tolerance=1e-8,
)

assert linear.tau == fixed_tau
assert generalized.tau == fixed_tau

In [ ]:
GridSolver = getattr(linear_solvers, "TikhonovOnAlphaGrid", None)
if GridSolver is None:
    GridSolver = linear_solvers.TikhonovAlphaGrid

def make_grid_solver(setting, parameters):
    signature = inspect.signature(GridSolver)
    arguments = signature.parameters

    if "regpars" in arguments:
        options = {"regpars": iter(parameters)}
    elif "alphas" in arguments:
        options = {"alphas": iter(parameters)}
    else:
        raise TypeError(
            f"Please adapt the parameter-sequence argument for "
            f"{GridSolver.__name__}{signature}"
        )

    if "cg_pars" in arguments:
        options["cg_pars"] = dict(cg_pars)
    else:
        # Earlier alpha-grid interfaces exposed the CG tolerance directly.
        if "tol_fac" in arguments:
            options["tol_fac"] = 1e-10
        elif "tol_fac_cg" in arguments:
            options["tol_fac_cg"] = 1e-10
        else:
            raise TypeError(
                f"Please specify tight inner-CG tolerances for "
                f"{GridSolver.__name__}{signature}"
            )
        if "max_CG_iter" in arguments:
            options["max_CG_iter"] = 1000

    if "logging_level" in arguments:
        options["logging_level"] = "WARNING"
    if "data" in arguments:
        options["data"] = data.copy()

    return GridSolver(setting, **options)

regpars = [1e-2, 5e-3, 2e-3]

irgnm = IrgnmCG(
    make_setting(),
    regpars=iter(regpars),
    cg_pars=dict(cg_pars),
    logging_level="WARNING",
)
alpha_grid = make_grid_solver(make_setting(), regpars)

compare(
    "IRGNM-CG versus Tikhonov alpha grid",
    take_steps(irgnm, 3),
    take_steps(alpha_grid, 3),
    tolerance=1e-6,
)

In [ ]:
alpha = 1e-2

levenberg_marquardt = LevenbergMarquardt(
    make_setting(),
    regpars=iter([alpha, alpha, alpha]),
    cg_pars=dict(cg_pars),
    logging_level="WARNING",
)
spectral_iterated = Spectral_IteratedTikhonov(
    make_setting(),
    regpar=alpha,
    k=3,
)

lm_history = take_steps(levenberg_marquardt, 3)
spectral_result = take_steps(spectral_iterated, 1)

compare(
    "Levenberg–Marquardt versus iterated Tikhonov",
    [lm_history[-1]],
    spectral_result,
    tolerance=1e-6,
)